# DL Model for echocardiogram synthetic data generation

## Encoder-decoder training

Importar las librerías necesarias

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
import lpips

/opt/miniconda3/envs/medim_torch/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /opt/miniconda3/envs/medim_torch/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


In [3]:
import cv2

In [4]:
import os
from pathlib import Path
import shutil

Herramienta para procesar videos en frames

In [5]:
def extract_frames_from_avi(video_path, output_folder):
    """
    Extract all frames from an AVI video file and save them as PNG images.
    
    Args:
        video_path (str): Path to the .avi video file
        output_folder (str): Folder where frames will be saved
        
    Returns:
        int: Number of frames extracted
    """
    # Create output folder if it doesn't exist
    Path(output_folder).mkdir(parents=True, exist_ok=True)
    
    # Open the video file
    cap = cv2.VideoCapture(video_path)
    
    if not cap.isOpened():
        raise ValueError(f"Could not open video file: {video_path}")
    
    frame_count = 0
    video_name = Path(video_path).stem  # Get filename without extension
    
    while True:
        ret, frame = cap.read()
        
        if not ret:
            break
        
        # Save the frame as PNG
        frame_filename = os.path.join(output_folder, f"{video_name}_frame_{frame_count:06d}.png")
        cv2.imwrite(frame_filename, frame)
        frame_count += 1
    
    cap.release()
    print(f"Extracted {frame_count} frames from {video_path} to {output_folder}")
    
    return frame_count

In [6]:
def process_avi_folder(input_folder, output_base_folder):
    """
    Process all .avi files in a folder, extracting frames from each video.
    Creates a subfolder for each video's frames.
    
    Args:
        input_folder (str): Folder containing .avi files
        output_base_folder (str): Base folder where subfolders for each video will be created
    
    Returns:
        dict: Dictionary with video names as keys and frame counts as values
    """
    Path(output_base_folder).mkdir(parents=True, exist_ok=True)
    
    # Find all .avi files in the input folder
    avi_files = sorted(Path(input_folder).glob("*.avi"))
    
    if not avi_files:
        print(f"No .avi files found in {input_folder}")
        return {}
    
    results = {}
    
    for video_file in avi_files:
        video_name = video_file.stem
        output_folder = os.path.join(output_base_folder, video_name)
        
        try:
            frame_count = extract_frames_from_avi(str(video_file), output_folder)
            results[video_name] = frame_count
        except Exception as e:
            print(f"Error processing {video_file}: {e}")
            results[video_name] = None
    
    print(f"\nProcessing complete! Summary:")
    for video_name, count in results.items():
        if count is not None:
            print(f"  {video_name}: {count} frames")
        else:
            print(f"  {video_name}: Failed")
    
    return results

In [ ]:
def consolidate_images_from_subfolders(input_folder, output_folder, move=False):
    """
    Consolidate all images from a folder with subfolders into a single folder.
    
    This function recursively searches through all subfolders and collects all image files
    (jpg, jpeg, png, bmp, tiff, gif) into a single output folder.
    
    Args:
        input_folder (str): Base folder containing subfolders with images
        output_folder (str): Destination folder where all images will be collected
        move (bool): If True, move files (cut). If False, copy files. Default is False.
    
    Returns:
        dict: Summary with 'total_files' count and 'files_by_subfolder' breakdown
    """
    # Define supported image extensions
    image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.gif', '.JPG', '.JPEG', '.PNG', '.BMP', '.TIFF', '.GIF'}
    
    # Create output folder if it doesn't exist
    Path(output_folder).mkdir(parents=True, exist_ok=True)
    
    input_path = Path(input_folder)
    
    if not input_path.exists():
        raise ValueError(f"Input folder does not exist: {input_folder}")
    
    total_files = 0
    files_by_subfolder = {}
    
    # Recursively search for all image files
    for file_path in input_path.rglob('*'):
        if file_path.is_file() and file_path.suffix in image_extensions:
            # Get the relative subfolder path for tracking
            relative_path = file_path.relative_to(input_path)
            subfolder = str(relative_path.parent)
            
            # Track files by subfolder
            if subfolder not in files_by_subfolder:
                files_by_subfolder[subfolder] = 0
            files_by_subfolder[subfolder] += 1
            
            # Create destination filename (preserve original name)
            output_file = os.path.join(output_folder, file_path.name)
            
            # Handle duplicate filenames by appending underscore + counter
            counter = 1
            base_name = file_path.stem
            extension = file_path.suffix
            while os.path.exists(output_file):
                output_file = os.path.join(output_folder, f"{base_name}_{counter}{extension}")
                counter += 1
            
            # Copy or move the file
            if move:
                shutil.move(str(file_path), output_file)
            else:
                shutil.copy2(str(file_path), output_file)
            
            total_files += 1
    
    # Print summary
    print(f"{'Moved' if move else 'Copied'} {total_files} image files to {output_folder}")
    if files_by_subfolder:
        print("\nFiles by subfolder:")
        for subfolder, count in sorted(files_by_subfolder.items()):
            print(f"  {subfolder}: {count} files")
    
    return {
        'total_files': total_files,
        'files_by_subfolder': files_by_subfolder,
        'move': move
    }

### Clases de redes

In [10]:
class VQGAN(nn.Module):
    def __init__(self, vocab_size=1024, embed_dim=256):
        super().__init__()
        # Encoder: Reducción f=16 (256x256 -> 16x16)
        # Usamos la lógica de ConvNeXt simplificada para consistencia
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 64, 4, stride=4, padding=0), 
            nn.ReLU(),
            nn.Conv2d(64, 128, 2, stride=2, padding=0),
            nn.ReLU(),
            nn.Conv2d(128, embed_dim, 1) # Proyección al espacio latente
        )

        self.codebook = nn.Embedding(vocab_size, embed_dim)
        self.codebook.weight.data.uniform_(-1.0 / vocab_size, 1.0 / vocab_size)

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(embed_dim, 128, 2, stride=2, padding=0),
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, stride=4, padding=0),
            nn.ReLU(),
            nn.Conv2d(64, 1, 3, padding=1),
            nn.Tanh()
        )

    def quantize(self, z):
        # z: [B, C, H, W]
        z_flattened = z.permute(0, 2, 3, 1).contiguous().view(-1, z.shape[1])
        
        # Distancia Euclidiana: (a-b)^2 = a^2 + b^2 - 2ab
        d = torch.sum(z_flattened**2, dim=1, keepdim=True) + \
            torch.sum(self.codebook.weight**2, dim=1) - \
            2 * torch.matmul(z_flattened, self.codebook.weight.t())
        
        indices = torch.argmin(d, dim=1)
        z_q = self.codebook(indices).view(z.shape[0], z.shape[2], z.shape[3], z.shape[1])
        z_q = z_q.permute(0, 3, 1, 2).contiguous()
        
        # Straight-through estimator (STE)
        z_q_ste = z + (z_q - z).detach() 
        return z_q_ste, indices, z_q

    def forward(self, x):
        z_e = self.encoder(x)
        z_q_ste, indices, z_q = self.quantize(z_e)
        x_hat = self.decoder(z_q_ste)
        return x_hat, indices, z_e, z_q

In [11]:
class PatchGANDiscriminator(nn.Module):
    def __init__(self, in_channels=1):
        super().__init__()
        def block(in_f, out_f, stride=2):
            return nn.Sequential(
                nn.Conv2d(in_f, out_f, 4, stride=stride, padding=1),
                nn.BatchNorm2d(out_f),
                nn.LeakyReLU(0.2, inplace=True)
            )
        self.model = nn.Sequential(
            block(in_channels, 64),
            block(64, 128),
            block(128, 256),
            nn.Conv2d(256, 1, 4, padding=1) # Salida Patch 30x30 aprox
        )

    def forward(self, x):
        return self.model(x)

In [12]:
class VQGANLoss(nn.Module):
    def __init__(self):
        super().__init__()
        # 'vgg' es el estándar porque es muy sensible a texturas médicas
        self.perceptual_loss = lpips.LPIPS(net='vgg').eval() 
        
        # Congelamos la red LPIPS (no queremos entrenarla, solo usarla de juez)
        for param in self.perceptual_loss.parameters():
            param.requires_grad = False

    def forward(self, x, x_hat):
        # x y x_hat deben estar en rango [-1, 1]
        return self.perceptual_loss(x, x_hat).mean()

### Entrenamiento

In [ ]:
from tqdm import tqdm

In [13]:
# TRAINING
device = torch.device("mps" if torch.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")

In [14]:
model = VQGAN().to(device)
discriminator = PatchGANDiscriminator().to(device)
perceptual_loss_fn = lpips.LPIPS(net='vgg').to(device).eval()


Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]


/opt/miniconda3/envs/medim_torch/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/miniconda3/envs/medim_torch/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /opt/miniconda3/envs/medim_torch/lib/python3.10/site-packages/lpips/weights/v0.1/vgg.pth


In [15]:
opt_ae = torch.optim.Adam(model.parameters(), lr=1e-4, betas=(0.5, 0.9))
opt_disc = torch.optim.Adam(discriminator.parameters(), lr=1e-4, betas=(0.5, 0.9))

In [16]:
# def train_epoch(dataloader):
#     model.train()
#     discriminator.train()
    
#     for x in dataloader:
#         x = x.to(device)
        
#         # ---------------------------------------
#         # 1. ENTRENAMIENTO DEL AUTOENCODER (G)
#         # ---------------------------------------
#         opt_ae.zero_grad()
        
#         x_hat, indices, z_e, z_q = model(x)
        
#         # A. Pérdida de Reconstrucción + Perceptual
#         rec_loss = F.l1_loss(x, x_hat)
#         p_loss = perceptual_loss_fn(x, x_hat).mean()
        
#         # B. Pérdida del Codebook (VQ Loss)
#         # mse(z_e detenido, z_q) actualiza el diccionario
#         # mse(z_e, z_q detenido) es la commitment loss (entrena al encoder)
#         codebook_loss = F.mse_loss(z_e.detach(), z_q) + 0.25 * F.mse_loss(z_e, z_q.detach())
        
#         # C. Pérdida Adversaria (Engañar al Juez)
#         logits_fake = discriminator(x_hat)
#         g_loss = -torch.mean(logits_fake) # Hinge loss simplificada
        
#         # Total Autoencoder Loss
#         # Ajustamos lambdas para balancear (ajustar según resultados en Purdue)
#         total_loss_ae = rec_loss + p_loss + codebook_loss + 0.1 * g_loss
        
#         total_loss_ae.backward()
#         opt_ae.step()
        
#         # ---------------------------------------
#         # 2. ENTRENAMIENTO DEL DISCRIMINADOR (D)
#         # ---------------------------------------
#         opt_disc.zero_grad()
        
#         logits_real = discriminator(x)
#         logits_fake = discriminator(x_hat.detach()) # Importante el DETACH
        
#         # Hinge Loss para el Discriminador
#         loss_real = torch.mean(F.relu(1. - logits_real))
#         loss_fake = torch.mean(F.relu(1. + logits_fake))
#         total_loss_disc = 0.5 * (loss_real + loss_fake)
        
#         total_loss_disc.backward()
#         opt_disc.step()

#     print(f"AE Loss: {total_loss_ae.item():.4f} | Disc Loss: {total_loss_disc.item():.4f}")

In [ ]:
def train_epoch(dataloader, epoch_idx):
    model.train()
    discriminator.train()
    
    # Configuramos la barra de progreso
    pbar = tqdm(dataloader, desc=f"Epoch {epoch_idx}")
    
    for x in pbar:
        x = x.to(device)

        # --- PASO 1: AUTOENCODER (EL CREADOR) ---
        opt_ae.zero_grad()
        x_hat, indices, z_e, z_q = model(x)

        # A. Anatomía y Diccionario
        decoding_loss = F.mse_loss(x, x_hat)
        codebook_loss = F.mse_loss(z_e.detach(), z_q) + 0.01 * F.mse_loss(z_e, z_q.detach())

        # B. Detalle (GAN): El AE quiere que el Juez diga que lo falso es REAL (1)
        logits_fake = discriminator(x_hat)
        discrimination_loss = F.binary_cross_entropy_with_logits(logits_fake, torch.ones_like(logits_fake))

        # Pérdida Total (Fase 1)
        total_loss_ae = decoding_loss + codebook_loss + 0.05 * discrimination_loss
        total_loss_ae.backward()
        opt_ae.step()


        # --- PASO 2: DISCRIMINADOR (EL JUEZ) ---
        opt_disc.zero_grad()

        # El Juez quiere aprender a diferenciar: Real=1, Falso=0
        logits_real = discriminator(x)
        logits_fake_d = discriminator(x_hat.detach()) # Detach para seguridad

        loss_real = F.binary_cross_entropy_with_logits(logits_real, torch.ones_like(logits_real))
        loss_fake = F.binary_cross_entropy_with_logits(logits_fake_d, torch.zeros_like(logits_fake_d))

        total_loss_disc = (loss_real + loss_fake) * 0.5
        total_loss_disc.backward()
        opt_disc.step()

        # ---------------------------------------
        # Actualización de la barra de progreso
        # ---------------------------------------
        # Calculamos cuántos tokens diferentes se usaron en este batch
        active_tokens = len(torch.unique(indices))
        
        pbar.set_postfix({
            "AE_Loss": f"{total_loss_ae.item():.4f}",
            "Disc_Loss": f"{total_loss_disc.item():.4f}",
            "Tokens": active_tokens
        })

Procesar los videos en imágenes

In [ ]:
# process_avi_folder("EchoNet-Dynamic/some videos", "EchoNet-Dynamic/Videos")

In [ ]:
# consolidate_images_from_subfolders("EchoNet-Dynamic/subframes", "EchoNet-Dynamic/sub all frames", move=True)

In [61]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import save_image

In [23]:
from PIL import Image

In [41]:
# 1. Parámetros de configuración (Aprovechando tus 96GB de RAM)
NUM_EPOCHS = 100
BATCH_SIZE = 32 # Puedes subirlo a 64 si los frames son de 128x128
LEARNING_RATE = 1e-4
SAVE_EVERY = 5 # Guardar checkpoint cada 5 épocas
CHECKPOINT_DIR = "checkpoints_proid"

In [42]:
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs("samples", exist_ok=True)

In [43]:
class EchoDataset(Dataset):
    def __init__(self, folder_path, img_size=256):
        self.folder_path = folder_path
        # Listamos todos los archivos de imagen (ajusta las extensiones si es necesario)
        self.image_files = [f for f in os.listdir(folder_path) if f.endswith(('.png', '.jpg', '.jpeg'))]
        
        self.transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.Grayscale(num_output_channels=1),
            transforms.ToTensor(), # Convierte a [0, 1]
            transforms.Normalize(mean=[0.5], std=[0.5]) # Convierte a [-1, 1]
        ])

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.folder_path, self.image_files[idx])
        image = Image.open(img_path).convert('L') # Abrir como escala de grises
        return self.transform(image)

In [47]:
path = "EchoNet-Dynamic/sub all frames" #Cambiar carpeta para hacer el entrenamiento con todas las imagenes

In [52]:
device

device(type='mps')

In [55]:
# 3. EL BLOQUE PROTECTOR (Indispensable en Mac)
if __name__ == '__main__':
    
    # Dataset y DataLoader
    dataset = EchoDataset(folder_path="EchoNet-Dynamic/sub all frames", img_size=256)
    train_loader = DataLoader(
        dataset, 
        batch_size=32, 
        shuffle=True, 
        num_workers=0
    )

In [62]:
print(f"Iniciando entrenamiento por {NUM_EPOCHS} épocas...")

for epoch in range(1, NUM_EPOCHS + 1):
    # Ejecutamos la función de época que definimos antes
    train_epoch(train_loader, epoch)
    
    # --- Tareas de mantenimiento al final de cada época ---
    
    # A. Guardar pesos del modelo
    if epoch % SAVE_EVERY == 0:
        checkpoint_path = os.path.join(CHECKPOINT_DIR, f"vqgan_heart_ep{epoch}.pth")
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'disc_state_dict': discriminator.state_dict(),
            'opt_ae_state_dict': opt_ae.state_dict(),
            'opt_disc_state_dict': opt_disc.state_dict(),
        }, checkpoint_path)
        print(f"Checkpoint guardado en: {checkpoint_path}")

    # B. Generar una muestra visual para control de calidad
    # Esto te permite ver si el modelo está aprendiendo los bordes correctamente
    with torch.no_grad():
        model.eval()
        # Tomamos un batch de ejemplo
        sample_x = next(iter(train_loader)).to(device)
        x_hat, _, _, _ = model(sample_x)
        
        # Guardamos la comparación (Opcional: puedes usar torchvision.utils.save_image)
        # transform.Normalize(mean=[0.5], std=[0.5]) invierte [-1, 1] a [0, 1]
        sample_img = (x_hat[0] + 1) / 2 
        save_image(sample_img, f"samples/res_ep{epoch}.png")
        model.train()

print("¡Entrenamiento completado!")

Iniciando entrenamiento por 100 épocas...


Epoch 5: 100%|██████████| 132/132 [02:01<00:00,  1.08it/s, AE_Loss=0.8222, Disc_Loss=0.7225, Tokens=30]


Checkpoint guardado en: checkpoints_proid/vqgan_heart_ep5.pth


Epoch 10: 100%|██████████| 132/132 [01:04<00:00,  2.04it/s, AE_Loss=0.8172, Disc_Loss=0.7230, Tokens=28]


Checkpoint guardado en: checkpoints_proid/vqgan_heart_ep10.pth


Epoch 13:  55%|█████▌    | 73/132 [00:43<00:35,  1.67it/s, AE_Loss=0.8121, Disc_Loss=0.7229, Tokens=32]


KeyboardInterrupt: 